# Notebook 01 - Data Cleaning & Validation

**Arbitrage-Free Implied Volatility Surface**

This notebook turns the raw WRDS/OptionMetrics CSV exports into a **clean, reproducible Parquet dataset** that every downstream notebook (SVI/SSVI benchmark, deep smoother, neural operator) builds on.

**Pipeline:**
1. Centralized configuration (paths, date format, filtering rules).
2. Typed loading of each table (zero curve, forwards, option quotes, OptionMetrics surface, historical volatility).
3. Feature engineering: `strike ÷ 1000`, date parsing, maturity `τ`, mid price, bid-ask spread, log-moneyness `k`.
4. A **principled missing-value strategy** (understand *why* each NaN exists, then act by category).
5. **Contract handling**: label SPX vs SPXW, resolve third-Friday collisions (one expiration = one contract).
6. Cross-validation: recomputed IV vs OptionMetrics IV over the whole dataset, temporal coverage, quote density.
7. Automated tests, then save clean Parquet files.

## 0. Imports & Configuration

In [ ]:
# --- imports ---
import os
from pathlib import Path
import polars as pl
import numpy as np
import plotly.express as px
from scipy.stats import norm

In [ ]:
# --- paths ---
def default_data_dir():
    env = os.environ.get("THESIS_DATA_DIR")
    if env:
        return Path(env)
    for candidate in [Path("data/raw"), Path("main/data/raw"), Path("data")]:
        if (candidate / "option_prices.csv").exists():
            return candidate
    return Path("data/raw")

DATA_DIR = default_data_dir()
OUT_DIR = Path(os.environ.get(
    "THESIS_OUT_DIR",
    str(DATA_DIR.parent / "clean" if DATA_DIR.name == "raw" else DATA_DIR / "clean"),
))
OUT_DIR.mkdir(parents=True, exist_ok=True)

In [17]:
# --- parameters ---
SEP           = ","
DATE_FMT      = "%d/%m/%Y"     
SECID_SPX     = 108105         # SPX security id in OptionMetrics
DTE_MIN       = 7              # drop maturities < 7 days (noise / edge effects)
DTE_MAX       = 730            # drop maturities > 2 years (illiquid)
KEEP_WEEKLYS  = True           # True: keep SPXW and de-duplicate 3rd-Friday collisions / False: standard SPX only (am_settlement == 1)
REPLACE_RAW   = False          # True: delete raw CSVs after a successful save (keep False until sure)
NULLS         = ["", "NA", "NaN", "nan", ".", "NULL"]

print("DATA_DIR:", DATA_DIR.resolve())
print("OUT_DIR :", OUT_DIR.resolve())
print("KEEP_WEEKLYS:", KEEP_WEEKLYS)

DATA_DIR: /Users/hamzaelarji/Desktop/All/Bureau/cours/Imperial College/Ze master thesis /main/data/raw
OUT_DIR : /Users/hamzaelarji/Desktop/All/Bureau/cours/Imperial College/Ze master thesis /main/data/clean
KEEP_WEEKLYS: True


In [18]:
# --- utilities ---
def to_date(col, fmt=DATE_FMT):
    """Parse a string column into dates; invalid values become null."""
    return pl.col(col).str.strptime(pl.Date, fmt, strict=False)

def collect_stream(lf):
    """Materialize a LazyFrame with the streaming engine."""
    try:
        return lf.collect(engine="streaming")
    except TypeError:
        return lf.collect(streaming=True)

def require_columns(path, expected):
    cols = set(pl.read_csv(path, n_rows=0, separator=SEP).columns)
    missing = sorted(set(expected) - cols)
    assert not missing, f"Missing columns in {path.name}: {missing}"

def null_report(df, name):
    n = df.height
    nc = df.null_count()
    rows = [(c, int(nc[c][0]), round(100 * int(nc[c][0]) / max(n, 1), 3))
            for c in df.columns if int(nc[c][0]) > 0]
    print(f"[{name}]  {n:,} rows, {df.width} columns")
    for c, k, pct in rows:
        print(f"    NaN  {c:<28} {k:>9,}  ({pct}%)")
    if not rows:
        print("    No missing values")
    return df

def check(condition, message):
    assert bool(condition), message
    print("OK -", message)

## 1. Zero-Coupon Yield Curve

Risk-free rate `r` per date and horizon, used to build the discount factor $D=e^{-r\tau}$ (needed to recompute IV independently as a validation). OptionMetrics reports the rate as an **annualized continuously-compounded percentage** — divide by 100 before use.

In [19]:
require_columns(DATA_DIR / "zero_coupon_yield_curve.csv", ["date", "days", "rate"])

zc = (
    pl.scan_csv(DATA_DIR / "zero_coupon_yield_curve.csv",
                separator=SEP, null_values=NULLS, try_parse_dates=False)
    .with_columns(
        to_date("date").alias("date"),
        pl.col("days").cast(pl.Int32, strict=False),
        pl.col("rate").cast(pl.Float64, strict=False),
    )
    .drop_nulls(["date", "days", "rate"])
    .filter(pl.col("days").is_between(DTE_MIN, DTE_MAX) & pl.col("rate").is_finite())
    .unique(subset=["date", "days"])
    .sort(["date", "days"])
)
zc = collect_stream(zc)
null_report(zc, "zero_curve")
print("Period:", zc["date"].min(), "->", zc["date"].max())
print("Days  :", zc["days"].min(), "->", zc["days"].max())
print("Rate %:", round(zc["rate"].min(), 3), "->", round(zc["rate"].max(), 3))


[zero_curve]  23,248 rows, 3 columns
    No missing values
Period: 2018-01-02 -> 2025-08-29
Days  : 7 -> 730
Rate %: 0.059 -> 5.793


In [20]:
dates = zc.select("date").unique().sort("date").to_series()
sel = dates.gather([int(i * (len(dates) - 1) / 9) for i in range(10)])
fig = px.line(zc.filter(pl.col("date").is_in(sel.to_list())).to_pandas(),
              x="days", y="rate", color="date", markers=True,
              title="Zero-coupon yield curves at selected dates",
              labels={"days": "Maturity (days)", "rate": "Rate (%)", "date": "Date"})
fig.show()


## 2. Forward Prices

Forward `F` per date and expiration. It is used to:
- (a) build log-moneyness $k=\ln(K/F)$, 
- (b) validate the `forward_price` already present in the quotes table,
- (c) and fill missing forwards in the quotes via a join.

In [21]:
require_columns(DATA_DIR / "forward_price.csv",
                ["secid", "date", "expiration", "AMSettlement", "ForwardPrice"])

fwd = (
    pl.scan_csv(DATA_DIR / "forward_price.csv",
                separator=SEP, null_values=NULLS, try_parse_dates=False)
    .with_columns(
        pl.col("secid").cast(pl.Int64, strict=False),
        to_date("date").alias("date"),
        to_date("expiration").alias("expiration"),
        pl.col("AMSettlement").cast(pl.Int8, strict=False),
        pl.col("ForwardPrice").cast(pl.Float64, strict=False),
    )
    .drop_nulls(["secid", "date", "expiration", "AMSettlement", "ForwardPrice"])
    .with_columns((pl.col("expiration") - pl.col("date")).dt.total_days().alias("dte"))
    .filter((pl.col("dte") >= DTE_MIN) & (pl.col("dte") <= DTE_MAX)
            & (pl.col("ForwardPrice") > 0) & pl.col("ForwardPrice").is_finite())
    .unique(subset=["secid", "date", "expiration", "AMSettlement"])
    .sort(["date", "expiration"])
)
fwd = collect_stream(fwd)
null_report(fwd, "forward")
print("Period :", fwd["date"].min(), "->", fwd["date"].max())
print("Forward:", round(fwd["ForwardPrice"].min(), 3), "->", round(fwd["ForwardPrice"].max(), 1))

# lazy view for the join in Section 3 (renamed keys + suffixed value column)
fwd_lf = fwd.lazy().select([
    "secid", "date",
    pl.col("expiration").alias("exdate"),
    pl.col("AMSettlement").alias("am_settlement"),
    pl.col("ForwardPrice").alias("forward_price_fwd"),
])


[forward]  82,681 rows, 6 columns
    No missing values
Period : 2018-01-02 -> 2025-08-29
Forward: 2237.699 -> 6867.0


## 3. Option Quotes — the main dataset

Cleaning happens **inside the lazy plan**, so filters shrink the data before materialization.

**Feature engineering:** `strike = strike_price/1000`; `dte = exdate - date`, `tau = dte/365` (ACT/365); `mid = (bid+ask)/2`, `spread = ask - bid`; `cp_flag → C/P`; `contract_type` from settlement.

**Quality filters:** `best_bid > 0`, `best_offer ≥ best_bid`, `DTE_MIN ≤ dte ≤ DTE_MAX`, `cp_flag ∈ {C,P}`, and **`iv_om` present** (we drop rows without an implied volatility — see the note below).

**Forward handling:** we keep the quote's own `forward_price` when present and fill the rest from the Forward table via a left join.

In [22]:
require_columns(DATA_DIR / "option_prices.csv",
                ["secid", "date", "exdate", "cp_flag", "strike_price",
                 "best_bid", "best_offer", "volume", "open_interest",
                 "impl_volatility", "am_settlement", "forward_price"])

OPT_COLS = ["secid", "date", "exdate", "cp_flag", "strike_price",
            "best_bid", "best_offer", "volume", "open_interest",
            "impl_volatility", "am_settlement", "forward_price"]

opt_lf = (
    pl.scan_csv(DATA_DIR / "option_prices.csv",
                separator=SEP, null_values=NULLS, try_parse_dates=False,
                infer_schema_length=20000)
    .select(OPT_COLS)
    .with_columns(
        pl.col("secid").cast(pl.Int64, strict=False),
        to_date("date").alias("date"),
        to_date("exdate").alias("exdate"),
        pl.col("cp_flag").str.strip_chars().str.to_uppercase(),
        (pl.col("strike_price").cast(pl.Float64, strict=False) / 1000.0).alias("strike"),
        pl.col("best_bid").cast(pl.Float64, strict=False),
        pl.col("best_offer").cast(pl.Float64, strict=False),
        pl.col("volume").cast(pl.Float64, strict=False),
        pl.col("open_interest").cast(pl.Float64, strict=False),
        pl.col("impl_volatility").cast(pl.Float64, strict=False).alias("iv_om"),
        pl.col("am_settlement").cast(pl.Int8, strict=False),
        pl.col("forward_price").cast(pl.Float64, strict=False),
    )
    # raw-missing flags (before any fill), used for the missing-value recap
    .with_columns(
        pl.col("volume").is_null().alias("volume_missing_raw"),
        pl.col("open_interest").is_null().alias("open_interest_missing_raw"),
        pl.col("forward_price").is_null().alias("forward_price_missing_raw"),
        pl.col("iv_om").is_null().alias("iv_om_missing_raw"),
    )
    # engineered fields
    .with_columns(
        (pl.col("exdate") - pl.col("date")).dt.total_days().alias("dte"),
        ((pl.col("best_bid") + pl.col("best_offer")) / 2).alias("mid"),
        (pl.col("best_offer") - pl.col("best_bid")).alias("spread"),
        pl.when(pl.col("am_settlement") == 1).then(pl.lit("SPX"))
          .when(pl.col("am_settlement") == 0).then(pl.lit("SPXW"))
          .otherwise(pl.lit("UNKNOWN")).alias("contract_type"),
        # volume / open_interest: missing = no activity = 0
        pl.col("volume").fill_null(0.0).cast(pl.Int64, strict=False),
        pl.col("open_interest").fill_null(0.0).cast(pl.Int64, strict=False),
    )
    .with_columns((pl.col("dte") / 365.0).alias("tau"))
    # quality filters
    .filter(
        (pl.col("secid") == SECID_SPX)
        & pl.col("date").is_not_null() & pl.col("exdate").is_not_null()
        & (pl.col("strike") > 0) & pl.col("strike").is_finite()
        & (pl.col("best_bid") > 0) & pl.col("best_bid").is_finite()
        & (pl.col("best_offer") >= pl.col("best_bid")) & pl.col("best_offer").is_finite()
        & (pl.col("dte") >= DTE_MIN) & (pl.col("dte") <= DTE_MAX)
        & pl.col("cp_flag").is_in(["C", "P"])
        & pl.col("iv_om").is_not_null()
    )
    # fill forward from the Forward table where missing
    .join(fwd_lf, on=["secid", "date", "exdate", "am_settlement"], how="left")
    .with_columns(
        (pl.col("forward_price_missing_raw") & pl.col("forward_price_fwd").is_not_null())
            .alias("forward_filled_from_table"),
        pl.coalesce(["forward_price", "forward_price_fwd"]).alias("forward_price"),
    )
    .drop("forward_price_fwd")
    .filter((pl.col("forward_price") > 0) & pl.col("forward_price").is_finite())
    # coordinates + OTM flag
    .with_columns((pl.col("strike") / pl.col("forward_price")).log().alias("k"))
    .with_columns(
        pl.when(pl.col("cp_flag") == "C").then(pl.col("k") >= 0)
          .otherwise(pl.col("k") < 0).alias("is_otm")
    )
)

# standard-only path: drop SPXW entirely
if not KEEP_WEEKLYS:
    opt_lf = opt_lf.filter(pl.col("am_settlement") == 1)

opt = collect_stream(opt_lf)
null_report(opt, "option_prices")


[option_prices]  28,671,274 rows, 26 columns
    No missing values


secid,date,exdate,cp_flag,strike_price,best_bid,best_offer,volume,open_interest,impl_volatility,am_settlement,forward_price,strike,iv_om,volume_missing_raw,open_interest_missing_raw,forward_price_missing_raw,iv_om_missing_raw,dte,mid,spread,contract_type,tau,forward_filled_from_table,k,is_otm
i64,date,date,str,i64,f64,f64,i64,i64,f64,i8,f64,f64,f64,bool,bool,bool,bool,i64,f64,f64,str,f64,bool,f64,bool
108105,2018-01-12,2018-06-15,"""C""",2025000,757.4,763.7,0,160,0.181767,1,2791.091602,2025.0,0.181767,false,false,true,false,154,760.55,6.3,"""SPX""",0.421918,true,-0.320863,false
108105,2018-01-12,2018-06-15,"""C""",2050000,733.0,739.3,0,1327,0.19395,1,2791.091602,2050.0,0.19395,false,false,true,false,154,736.15,6.3,"""SPX""",0.421918,true,-0.308593,false
108105,2018-01-12,2018-06-15,"""C""",2075000,708.7,715.0,0,1201,0.200548,1,2791.091602,2075.0,0.200548,false,false,true,false,154,711.85,6.3,"""SPX""",0.421918,true,-0.296472,false
108105,2018-01-12,2018-06-15,"""C""",2100000,684.3,690.6,0,2608,0.201245,1,2791.091602,2100.0,0.201245,false,false,true,false,154,687.45,6.3,"""SPX""",0.421918,true,-0.284495,false
108105,2018-01-12,2018-06-15,"""C""",2125000,660.0,666.3,0,3007,0.201756,1,2791.091602,2125.0,0.201756,false,false,true,false,154,663.15,6.3,"""SPX""",0.421918,true,-0.272661,false
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
108105,2025-08-27,2025-10-06,"""P""",6625000,154.7,161.4,0,0,0.101885,0,6508.06018,6625.0,0.101885,false,false,true,false,40,158.05,6.7,"""SPXW""",0.109589,true,0.017809,false
108105,2025-08-27,2025-10-06,"""P""",6650000,171.7,177.7,0,0,0.099882,0,6508.06018,6650.0,0.099882,false,false,true,false,40,174.7,6.0,"""SPXW""",0.109589,true,0.021575,false
108105,2025-08-27,2025-10-06,"""P""",6700000,206.7,216.0,0,1,0.095957,0,6508.06018,6700.0,0.095957,false,false,true,false,40,211.35,9.3,"""SPXW""",0.109589,true,0.029066,false


**Missing implied volatility.** We remove all observations with missing `impl_volatility`. In the raw dataset, `impl_volatility`, `delta`, and `vega` are missing on exactly the same rows (approximately 5.7%). Whenever IV is unavailable, the Greeks are also missing, and the reverse never occurs. Consequently, the missing Greeks cannot be recovered from an existing IV. Around 99.98% of these missing observations correspond to **deep in-the-money (ITM)** options, for which OptionMetrics does not report IV because the option price is almost entirely intrinsic value. In these cases, the remaining time value is too small relative to the bid-ask spread for a reliable inversion of the Black-Scholes model. Only about 118 out of 8.5 million missing observations are out-of-the-money (OTM). Since our analysis focuses on OTM options and this number is negligible, we simply remove all rows with missing IV rather than attempting to recompute the missing Greeks.

**Contract types: SPX versus SPXW.** Standard SPX options (third-Friday expirations, AM-settled, `am_settlement = 1`) and SPXW weekly options (PM-settled, `am_settlement = 0`) can share the same expiration date while trading at slightly different prices. Combining both contract types would therefore produce multiple implied volatilities for the same strike-maturity pair `(k, \tau)`, introducing duplicate observations within a volatility slice. Since the `root` column is empty in this dataset, we distinguish the two contract families using the settlement flag. We retain both standard and weekly contracts because the weekly series provides denser coverage at short maturities. However, when both contract types expire on the same date (the third Friday of the month), we retain only the standard SPX contract. This ensures that each expiration date corresponds to a single contract while preserving the additional maturity coverage provided by SPXW on all other expiration dates.


In [23]:
# --- Resolve SPX/SPXW collisions: one date -> a single contract ---
if KEEP_WEEKLYS:
    before = opt.height
    opt = (
        opt.with_columns(
            pl.when(pl.col("contract_type") == "SPX").then(0).otherwise(1).alias("_pref")
        )
        .with_columns(pl.col("_pref").min().over(["date", "exdate"]).alias("_best"))
        .filter(pl.col("_pref") == pl.col("_best"))
        .drop("_pref", "_best")
    )
    print(f"SPX/SPXW de-duplication: {before:,} -> {opt.height:,} rows")

dup = (opt.group_by(["date", "exdate"])
          .agg(pl.col("contract_type").n_unique().alias("n"))
          .filter(pl.col("n") > 1))
print("remaining collisions:", dup.height)
print(opt.group_by("contract_type").agg(pl.len().alias("n")).sort("n", descending=True))

otm = opt.filter(pl.col("is_otm"))


SPX/SPXW de-duplication: 28,671,274 -> 23,746,235 rows
remaining collisions: 0 (must be 0)
shape: (2, 2)
┌───────────────┬──────────┐
│ contract_type ┆ n        │
│ ---           ┆ ---      │
│ str           ┆ u32      │
╞═══════════════╪══════════╡
│ SPXW          ┆ 14295583 │
│ SPX           ┆ 9450652  │
└───────────────┴──────────┘


## 4. Discount factor + IV validation (whole dataset)

We attach the zero-curve rate at the nearest horizon per date (`join_asof`), giving $D=e^{-r\tau}$. Then we **recompute the implied volatility ourselves** (vectorized bisection on Black-76) for **every** quote and compare it to `iv_om`. A small median gap validates the full chain: forward, discount, coordinates, inversion.

In [24]:
def attach_rate(df, zc):
    if "rate" in df.columns:
        df = df.drop("rate")
    left = df.sort(["date", "dte"])
    right = zc.rename({"days": "dte"}).select(["date", "dte", "rate"]).sort(["date", "dte"])
    return left.join_asof(right, on="dte", by="date", strategy="nearest")

opt = attach_rate(opt, zc)
opt = opt.with_columns(
    (-(pl.col("rate") / 100.0) * pl.col("tau")).exp().alias("discount"),
).with_columns(
    (pl.col("discount").is_not_null() & pl.col("discount").is_finite()
     & (pl.col("discount") > 0)).alias("discount_valid"),
    pl.col("rate").is_null().alias("rate_missing_after_join"),
)
otm = opt.filter(pl.col("is_otm"))

print("rate NaN after join:", int(opt["rate_missing_after_join"].sum()))
print("discount:", round(float(opt["discount"].min()), 4), "->", round(float(opt["discount"].max()), 4))


/var/folders/th/b_cdn0h107lcl7b1cp9xd7zc0000gn/T/ipykernel_54458/1491036354.py:6: UserWarning:

Sortedness of columns cannot be checked when 'by' groups provided



rate NaN after join: 0
discount: 0.9007 -> 1.0


In [25]:
# --- vectorized Black-76 + implied vol (bisection) for validation ---
def bs_vec(F, K, tau, sig, D, is_call):
    sq = np.sqrt(tau)
    d1 = (np.log(F / K) + 0.5 * sig * sig * tau) / (sig * sq)
    d2 = d1 - sig * sq
    call = D * (F * norm.cdf(d1) - K * norm.cdf(d2))
    put  = D * (K * norm.cdf(-d2) - F * norm.cdf(-d1))
    return np.where(is_call, call, put)

def implied_vol_vec(price, F, K, tau, D, is_call, lo=1e-6, hi=5.0, iters=60):
    price, F, K, tau, D = (np.asarray(x, float) for x in (price, F, K, tau, D))
    is_call = np.asarray(is_call, bool)
    intr = np.where(is_call, D * np.maximum(F - K, 0.0), D * np.maximum(K - F, 0.0))
    cap  = np.where(is_call, D * F, D * K)
    valid = (np.isfinite(price) & np.isfinite(F) & np.isfinite(K) & np.isfinite(tau)
             & np.isfinite(D) & (tau > 0) & (F > 0) & (K > 0) & (D > 0)
             & (price >= intr - 1e-8) & (price <= cap + 1e-8))
    p = np.where(valid, price, np.nan)
    lo = np.full(price.shape, lo); hi = np.full(price.shape, hi)
    for _ in range(iters):
        mid = 0.5 * (lo + hi)
        hi = np.where(bs_vec(F, K, tau, mid, D, is_call) - p > 0, mid, hi)
        lo = np.where(bs_vec(F, K, tau, mid, D, is_call) - p > 0, lo, mid)
    iv = 0.5 * (lo + hi); iv[~valid] = np.nan
    return iv

val = opt.filter(pl.col("discount_valid") & pl.col("mid").is_finite())
iv_ours = implied_vol_vec(val["mid"].to_numpy(), val["forward_price"].to_numpy(),
                          val["strike"].to_numpy(), val["tau"].to_numpy(),
                          val["discount"].to_numpy(), (val["cp_flag"] == "C").to_numpy())
iv_om = val["iv_om"].to_numpy()
diff = np.abs(iv_ours - iv_om); m = np.isfinite(diff)

print(f"compared: {m.sum():,} / {len(diff):,}   (not inverted: {(~m).sum():,})")
print(f"|IV_ours - IV_OM|  median {np.median(diff[m]):.5f}  "
      f"90% {np.quantile(diff[m], .9):.5f}  99% {np.quantile(diff[m], .99):.5f}")
print(f"share < 0.001: {100*np.mean(diff[m] < 1e-3):.2f}%   < 0.01: {100*np.mean(diff[m] < 1e-2):.2f}%")
is_otm = val["is_otm"].to_numpy()
for lab, msk in [("OTM", is_otm & m), ("ITM", (~is_otm) & m)]:
    if msk.sum():
        print(f"  {lab:3}: median {np.median(diff[msk]):.5f}  99% {np.quantile(diff[msk], .99):.5f}  (n={msk.sum():,})")


compared: 23,741,053 / 23,746,235   (not inverted: 5,182)
|IV_ours - IV_OM|  median 0.00003  90% 0.00207  99% 0.01687
share < 0.001: 81.40%   < 0.01: 98.04%
  OTM: median 0.00000  99% 0.01409  (n=12,378,168)
  ITM: median 0.00009  99% 0.02047  (n=11,362,885)


## 5. OptionMetrics volatility surface - industry benchmark

Not training data: the surface OptionMetrics already smoothed on a standardized delta×maturity grid. Kept clean for the final comparison (*our method vs the proprietary smoother*).

In [26]:
require_columns(DATA_DIR / "volatility_surface.csv",
                ["secid", "date", "days", "delta", "impl_volatility",
                 "impl_strike", "impl_premium", "dispersion", "cp_flag"])
vs = (
    pl.scan_csv(DATA_DIR / "volatility_surface.csv",
                separator=SEP, null_values=NULLS, try_parse_dates=False)
    .with_columns(
        pl.col("secid").cast(pl.Int64, strict=False),
        to_date("date").alias("date"),
        pl.col("days").cast(pl.Int32, strict=False),
        pl.col("delta").cast(pl.Float64, strict=False),
        pl.col("impl_volatility").cast(pl.Float64, strict=False),
        pl.col("impl_strike").cast(pl.Float64, strict=False),
        pl.col("impl_premium").cast(pl.Float64, strict=False),
        pl.col("dispersion").cast(pl.Float64, strict=False),
        pl.col("cp_flag").str.strip_chars().str.to_uppercase(),
    )
    .drop_nulls(["date", "days", "delta", "impl_volatility"])
    .filter((pl.col("secid") == SECID_SPX) & (pl.col("days") > 0)
            & (pl.col("impl_volatility") > 0) & pl.col("impl_volatility").is_finite())
    .sort(["date", "days", "delta"])
)
vs = collect_stream(vs)
null_report(vs, "volatility_surface")


[volatility_surface]  713,966 rows, 11 columns
    No missing values


secid,date,days,delta,impl_volatility,impl_strike,impl_premium,dispersion,cp_flag,ticker,index_flag
i64,date,i32,f64,f64,f64,f64,f64,str,str,i64
108105,2018-01-02,10,-90.0,0.085995,2745.88,51.76049,0.042292,"""P""","""SPX""",1
108105,2018-01-02,10,-85.0,0.072183,2729.728,36.29908,0.031911,"""P""","""SPX""",1
108105,2018-01-02,10,-80.0,0.061748,2719.358,26.50636,0.018507,"""P""","""SPX""",1
108105,2018-01-02,10,-75.0,0.057698,2713.465,21.37625,0.007872,"""P""","""SPX""",1
108105,2018-01-02,10,-70.0,0.057399,2709.503,18.44526,0.003502,"""P""","""SPX""",1
…,…,…,…,…,…,…,…,…,…,…
108105,2025-08-29,730,70.0,0.196282,6097.788,1070.659,0.017096,"""C""","""SPX""",1
108105,2025-08-29,730,75.0,0.209029,5792.901,1287.835,0.021474,"""C""","""SPX""",1
108105,2025-08-29,730,80.0,0.224283,5428.501,1560.411,0.029116,"""C""","""SPX""",1


## 6. Historical volatility

Realized volatility computed by OptionMetrics per horizon. Not essential — useful only for the introduction (IV vs realized vol, the COVID spike).

In [27]:
require_columns(DATA_DIR / "historical_volatility.csv", ["secid", "date", "days", "volatility"])
hv = (
    pl.scan_csv(DATA_DIR / "historical_volatility.csv",
                separator=SEP, null_values=NULLS, try_parse_dates=False)
    .with_columns(
        pl.col("secid").cast(pl.Int64, strict=False),
        to_date("date").alias("date"),
        pl.col("days").cast(pl.Int32, strict=False),
        pl.col("volatility").cast(pl.Float64, strict=False),
    )
    .drop_nulls(["date", "days", "volatility"])
    .filter((pl.col("secid") == SECID_SPX) & (pl.col("days") > 0)
            & (pl.col("volatility") >= 0) & pl.col("volatility").is_finite())
    .sort(["date", "days"])
)
hv = collect_stream(hv)
null_report(hv, "historical_volatility")


[historical_volatility]  23,112 rows, 6 columns
    No missing values


secid,date,days,volatility,ticker,index_flag
i64,date,i32,f64,str,i64
108105,2018-01-02,10,0.077845,"""SPX""",1
108105,2018-01-02,14,0.060526,"""SPX""",1
108105,2018-01-02,30,0.061742,"""SPX""",1
108105,2018-01-02,60,0.063599,"""SPX""",1
108105,2018-01-02,91,0.057538,"""SPX""",1
…,…,…,…,…,…
108105,2025-08-29,182,0.233901,"""SPX""",1
108105,2025-08-29,273,0.207544,"""SPX""",1
108105,2025-08-29,365,0.189423,"""SPX""",1


## 7. Cross-checks & diagnostics

Temporal coverage, the missing-value policy recap, quote density over time, and a preview of clean smiles (**one slice = one expiration**, grouped by `exdate`).

In [28]:
# (a) temporal coverage
for name, d in [("options", opt), ("forward", fwd), ("zero", zc),
                ("vol_surface", vs), ("hist_vol", hv)]:
    print(f"{name:<12} {d['date'].min()}  ->  {d['date'].max()}   ({d['date'].n_unique()} dates)")

# (b) missing-value policy recap
print("\nMissing-value policy recap")
print(opt.select([
    pl.col("volume_missing_raw").sum().alias("volume_filled_zero"),
    pl.col("open_interest_missing_raw").sum().alias("open_interest_filled_zero"),
    pl.col("forward_price_missing_raw").sum().alias("forward_missing_in_quotes"),
    pl.col("forward_filled_from_table").sum().alias("forward_filled_from_table"),
]))


options      2018-01-02  ->  2025-08-29   (1926 dates)
forward      2018-01-02  ->  2025-08-29   (1926 dates)
zero         2018-01-02  ->  2025-08-29   (1926 dates)
vol_surface  2018-01-02  ->  2025-08-29   (1909 dates)
hist_vol     2018-01-02  ->  2025-08-29   (1926 dates)

Missing-value policy recap
shape: (1, 4)
┌────────────────────┬─────────────────────────┬─────────────────────────┬─────────────────────────┐
│ volume_filled_zero ┆ open_interest_filled_ze ┆ forward_missing_in_quot ┆ forward_filled_from_tab │
│ ---                ┆ ro                      ┆ es                      ┆ le                      │
│ u32                ┆ ---                     ┆ ---                     ┆ ---                     │
│                    ┆ u32                     ┆ u32                     ┆ u32                     │
╞════════════════════╪═════════════════════════╪═════════════════════════╪═════════════════════════╡
│ 0                  ┆ 0                       ┆ 23746235                ┆ 23

In [29]:
# (c) quote density over time
per_day = otm.group_by("date").agg(pl.len().alias("n")).sort("date")
print("OTM quotes/day:  min", int(per_day["n"].min()),
      " median", int(per_day["n"].median()), " max", int(per_day["n"].max()))
fig = px.line(per_day.to_pandas(), x="date", y="n",
              title="Number of OTM quotes per day",
              labels={"date": "Date", "n": "# quotes"})
fig.show()


OTM quotes/day:  min 3199  median 6429  max 10000


In [30]:
# (d) clean smiles on the busiest day, one slice per expiration
best_day = per_day.sort("n", descending=True)["date"][0]
day = otm.filter(pl.col("date") == best_day)
exdates = (day.group_by("exdate")
              .agg(pl.len().alias("n"), pl.col("tau").first().alias("tau"))
              .filter((pl.col("tau") * 365 >= 20) & (pl.col("n") >= 6))
              .sort("tau"))
rows = []
if exdates.height:
    idx = np.linspace(0, exdates.height - 1, min(5, exdates.height)).round().astype(int)
    for ex in exdates["exdate"].gather(idx.tolist()):
        s = day.filter(pl.col("exdate") == ex).sort("k")
        lab = f"{s['tau'][0]*365:.0f}d"
        for kk, iv in zip(s["k"].to_list(), s["iv_om"].to_list()):
            rows.append({"k": kk, "iv": iv, "maturity": lab})
if rows:
    fig = px.line(pl.DataFrame(rows).to_pandas(), x="k", y="iv", color="maturity",
                  markers=True, title=f"OptionMetrics smiles — {best_day}",
                  labels={"k": "log-moneyness k", "iv": "IV", "maturity": "maturity"})
    fig.show()
else:
    print("Not enough exploitable slices on", best_day)


## 8. Automated tests before saving

Turns the diagnostics into reproducible guardrails: if a cleaning assumption breaks (missing columns, unresolved forwards, inconsistent quotes, remaining collisions), the notebook stops before writing a misleading Parquet.

In [31]:
FINAL_OPT_COLS = [
    "date", "exdate", "dte", "tau",
    "cp_flag", "contract_type",
    "strike", "forward_price", "k", "is_otm",
    "best_bid", "best_offer", "mid", "spread",
    "volume", "open_interest",
    "iv_om", "rate", "discount",
    "volume_missing_raw", "open_interest_missing_raw",
    "forward_price_missing_raw", "discount_valid",
]

_missing = sorted(set(FINAL_OPT_COLS) - set(opt.columns))
check(not _missing, f"all final columns present (missing: {_missing})")
check(opt.height > 0, "options dataset is not empty")
check(otm.height > 0, "OTM subset is not empty")
check(opt.select(pl.col("secid").n_unique()).item() == 1 and opt["secid"][0] == SECID_SPX,
      "only the expected SPX secid is retained")
check(opt.filter(pl.col("cp_flag").is_in(["C", "P"]).not_()).height == 0,
      "cp_flag standardized to C/P")
check(opt.filter((pl.col("dte") < DTE_MIN) | (pl.col("dte") > DTE_MAX)).height == 0,
      "maturities satisfy the DTE filter")
check(opt.filter((pl.col("best_bid") <= 0) | (pl.col("best_offer") < pl.col("best_bid"))).height == 0,
      "bid/ask quotes are consistent")
check(opt.filter((pl.col("strike") <= 0) | (pl.col("forward_price") <= 0)
                 | pl.col("k").is_finite().not_()).height == 0,
      "strike, forward and k are valid")
check(int(opt["iv_om"].null_count()) == 0, "every retained row has an implied volatility")
check(int(opt["volume"].null_count()) == 0 and int(opt["open_interest"].null_count()) == 0,
      "volume/open_interest filled with zeros")
check(int(opt["forward_price"].null_count()) == 0, "forward_price complete after the join")
check(int(opt["is_otm"].null_count()) == 0, "OTM indicator complete")
check(opt.filter(pl.col("discount_valid").not_()).height == 0, "discount present, positive and finite")

_dup = (opt.group_by(["date", "exdate"]).agg(pl.col("contract_type").n_unique().alias("n"))
           .filter(pl.col("n") > 1))
check(_dup.height == 0, "one expiration = one contract (no SPX/SPXW collision)")
check(vs.height > 0 and hv.height > 0 and zc.height > 0 and fwd.height > 0,
      "auxiliary tables are not empty")


OK - all final columns present (missing: [])
OK - options dataset is not empty
OK - OTM subset is not empty
OK - only the expected SPX secid is retained
OK - cp_flag standardized to C/P
OK - maturities satisfy the DTE filter
OK - bid/ask quotes are consistent
OK - strike, forward and k are valid
OK - every retained row has an implied volatility
OK - volume/open_interest filled with zeros
OK - forward_price complete after the join
OK - OTM indicator complete
OK - discount present, positive and finite
OK - one expiration = one contract (no SPX/SPXW collision)
OK - auxiliary tables are not empty


## 9. Save the clean dataset (Parquet)

The main file keeps only what modeling needs: coordinates (`k`, `tau`), quotes (`mid`, `spread`, `best_bid/offer`), `iv_om`, `forward_price`, `discount`, the `is_otm` and `contract_type` labels, and a few diagnostic flags. **Greeks and technical ids are dropped** — Greeks are derivable from IV and can be recomputed after calibration. Raw CSVs are kept unless `REPLACE_RAW=True`.

In [32]:
opt_out = opt.select(FINAL_OPT_COLS)

targets = {
    "option_prices_clean.parquet": opt_out,
    "forward_clean.parquet": fwd,
    "zero_curve_clean.parquet": zc,
    "volatility_surface_clean.parquet": vs,
    "historical_volatility_clean.parquet": hv,
}
for fname, df in targets.items():
    path = OUT_DIR / fname
    df.write_parquet(path)
    print(f"written  {path}   ({df.height:,} rows, {path.stat().st_size/1e6:.2f} MB)")

if REPLACE_RAW:
    for csv in ["option_prices.csv", "forward_price.csv", "zero_coupon_yield_curve.csv",
                "volatility_surface.csv", "historical_volatility.csv"]:
        p = DATA_DIR / csv
        if p.exists():
            p.unlink(); print("removed raw:", p)


written  data/clean/option_prices_clean.parquet   (23,746,235 rows, 535.45 MB)
written  data/clean/forward_clean.parquet   (82,681 rows, 0.68 MB)
written  data/clean/zero_curve_clean.parquet   (23,248 rows, 0.17 MB)
written  data/clean/volatility_surface_clean.parquet   (713,966 rows, 13.19 MB)
written  data/clean/historical_volatility_clean.parquet   (23,112 rows, 0.14 MB)
